# Task 3 & 4: Model Determination and Training

**Project:** Drug Review Classification — Condition Prediction  
**Objective:** Determine the best model architecture and train it to classify patient conditions (Depression, High Blood Pressure, Type 2 Diabetes) from drug review text.

---

## Overview

This notebook covers:
1. Algorithm selection rationale
2. Individual model comparison (Task 3 — Model Determination)
3. Ensemble architecture design
4. Feature pipeline construction
5. Model training (Task 4)
6. Evaluation — accuracy, precision, recall, F1
7. Confusion matrix analysis
8. Cross-validation
9. Summary of findings

## 1. Setup

In [ ]:
import sys
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import joblib
import time

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, MaxAbsScaler
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, precision_recall_fscore_support
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

# Add project root so we can import custom transformers
project_root = Path('..').resolve()
sys.path.insert(0, str(project_root))
from web.services.custom_transformers import (
    TextFeatureExtractor, SentimentFeatureExtractor, LearnedVocabularyExtractor
)

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
PALETTE = ['#14b8a6', '#eaa0a2', '#67cbdc', '#3a4664', '#cbcbcb']

print('Setup complete.')

In [ ]:
# Load cleaned data
df_train = pd.read_csv('../data/processed/cleaned_train_data.csv')
df_test  = pd.read_csv('../data/processed/cleaned_test_data.csv')

X_train = df_train['review'].fillna('').values
y_train = df_train['condition'].values
X_test  = df_test['review'].fillna('').values
y_test  = df_test['condition'].values

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc  = label_encoder.transform(y_test)

print(f'Training samples : {len(X_train):,}')
print(f'Test samples     : {len(X_test):,}')
print(f'Classes          : {label_encoder.classes_.tolist()}')

## 2. Algorithm Selection Rationale (Task 3)

Text classification tasks require algorithms that can handle **high-dimensional sparse feature spaces** (TF-IDF produces thousands of features) while remaining robust to noisy, variable-length text. The following five algorithms were evaluated:

| Algorithm | Why evaluated | Known strengths for text | Known weaknesses |
|-----------|--------------|--------------------------|------------------|
| **Logistic Regression** | Strong linear baseline; fast; interpretable | Excellent on high-dim TF-IDF; well-calibrated probabilities | Assumes linear separability |
| **Naive Bayes (Multinomial)** | Classic text classification baseline | Very fast; works well on sparse data | Strong independence assumption rarely holds |
| **Linear SVC** | State-of-art for text in literature | Maximises margin; handles high dimensions | No probability output; sensitive to scaling |
| **Random Forest** | Non-linear; handles mixed features | Robust; feature importance available | Can overfit; slow on high dimensions |
| **XGBoost** | Boosted trees; best single model in most NLP benchmarks | Handles non-linearity; built-in regularisation | Slower to train; many hyperparameters |

All models are evaluated on the **same TF-IDF features** to ensure a fair comparison.

## 3. Individual Model Comparison

In [ ]:
# Shared TF-IDF vectoriser for fair comparison
tfidf_base = TfidfVectorizer(
    max_features=2000, stop_words='english',
    ngram_range=(1, 2), sublinear_tf=True
)

models = {
    'Logistic Regression': LogisticRegression(
        C=1.0, class_weight='balanced', max_iter=1000, random_state=42),
    'Naive Bayes': MultinomialNB(alpha=0.1),
    'Linear SVC': LinearSVC(
        C=1.0, class_weight='balanced', max_iter=2000, random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=15, class_weight='balanced',
        n_jobs=-1, random_state=42),
    'XGBoost': XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        eval_metric='mlogloss', n_jobs=-1, random_state=42,
        use_label_encoder=False),
}

results = []
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

print('Evaluating individual models (3-fold CV on TF-IDF features)...\n')

X_train_tfidf = tfidf_base.fit_transform(X_train)
X_test_tfidf  = tfidf_base.transform(X_test)

for name, model in models.items():
    t0 = time.time()

    # CV score
    # LinearSVC has no predict_proba — use decision_function scoring
    cv_scores = cross_val_score(model, X_train_tfidf, y_train_enc,
                                cv=cv, scoring='accuracy', n_jobs=-1)

    # Test performance
    model.fit(X_train_tfidf, y_train_enc)
    y_pred = model.predict(X_test_tfidf)
    test_acc = accuracy_score(y_test_enc, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_test_enc, y_pred, average='weighted'
    )
    elapsed = time.time() - t0

    results.append({
        'Model': name,
        'CV Mean': round(cv_scores.mean(), 4),
        'CV Std': round(cv_scores.std(), 4),
        'Test Accuracy': round(test_acc, 4),
        'Precision': round(p, 4),
        'Recall': round(r, 4),
        'F1 Score': round(f1, 4),
        'Time (s)': round(elapsed, 1),
    })
    print(f'  {name:<22}  CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}  '
          f'Test={test_acc:.4f}  F1={f1:.4f}  ({elapsed:.1f}s)')

results_df = pd.DataFrame(results).sort_values('Test Accuracy', ascending=False)
results_df = results_df.reset_index(drop=True)
results_df.index += 1
print('\n=== INDIVIDUAL MODEL COMPARISON ===')
results_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

model_names = results_df['Model'].tolist()
colors = [PALETTE[0] if i == 0 else '#cbd5e1' for i in range(len(model_names))]

# Test accuracy comparison
bars = axes[0].barh(model_names, results_df['Test Accuracy'],
                    color=colors, edgecolor='white', height=0.55)
axes[0].set_xlim(0.85, 1.0)
axes[0].set_title('Test Accuracy — Individual Models', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Accuracy', fontsize=11)
for bar, val in zip(bars, results_df['Test Accuracy']):
    axes[0].text(val + 0.001, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=10, fontweight='bold')
axes[0].invert_yaxis()

# CV mean with error bars
axes[1].barh(model_names, results_df['CV Mean'],
             xerr=results_df['CV Std'], color=colors,
             edgecolor='white', height=0.55, capsize=4)
axes[1].set_xlim(0.85, 1.0)
axes[1].set_title('3-Fold CV Accuracy ± Std', fontsize=13, fontweight='bold')
axes[1].set_xlabel('CV Accuracy', fontsize=11)
axes[1].invert_yaxis()

plt.suptitle('Individual Model Performance Comparison', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: outputs/figures/model_comparison.png')

### Individual Model Findings

From the comparison:

- **XGBoost** achieves the highest single-model accuracy — gradient boosting iteratively corrects errors made by previous trees, making it particularly effective at capturing complex patterns in review language
- **Logistic Regression** performs surprisingly well given its simplicity, validating that TF-IDF features are highly linearly separable for this task
- **Naive Bayes** performs the worst despite being a traditional text classifier — the independence assumption breaks down when review language contains correlated medical terms
- **Random Forest** and **Linear SVC** fall in the middle, each with distinct strengths

**Why not use XGBoost alone?** A single model's predictions carry uncertainty. Different algorithms make different types of errors — XGBoost may misclassify a Depression review that Logistic Regression identifies correctly, and vice versa. An ensemble exploits this diversity.

## 4. Ensemble Architecture — Design Rationale

### Why a Soft Voting Ensemble?

A **soft voting ensemble** combines the class probability estimates from multiple models, weighting each model's contribution. This is superior to hard voting (majority vote) because:

1. **Probability information is preserved** — a model that is 99% confident overrides one that is 51% confident
2. **Error diversity is exploited** — models that make errors on different samples combine to produce a stronger overall result
3. **Uncertainty is reduced** — disagreement between models lowers the overall confidence, which feeds into the OOD detection system

### Model Weights

Models are assigned weights proportional to their individual performance:

| Model | Weight | Justification |
|-------|--------|---------------|
| Logistic Regression | 1 | Reliable linear baseline |
| Random Forest | 2 | Strong non-linear performance |
| XGBoost | 3 | Highest individual accuracy |

### Feature Pipeline

Four complementary feature sets are combined using `FeatureUnion`:

| Feature set | Description | Captures |
|-------------|-------------|----------|
| **TF-IDF** | Term frequency-inverse document frequency on review text | Condition-specific vocabulary |
| **TextFeatureExtractor** | Character count, word count, sentence length, TTR | Writing style and review depth |
| **SentimentFeatureExtractor** | VADER compound, positive, negative, neutral scores | Emotional tone of the review |
| **LearnedVocabularyExtractor** | Condition-specific vocabulary learned from data | Discriminative medical terms |

## 5. Build and Train the Full Ensemble Pipeline

In [ ]:
# Build the full feature pipeline
tfidf = TfidfVectorizer(
    max_features=2000, stop_words='english',
    ngram_range=(1, 2), min_df=3, max_df=0.85, sublinear_tf=True
)

feature_union = FeatureUnion([
    ('tfidf',        tfidf),
    ('text_stats',   TextFeatureExtractor()),
    ('sentiment',    SentimentFeatureExtractor()),
    ('learned_vocab', LearnedVocabularyExtractor(max_features_per_class=50)),
])

ensemble = VotingClassifier(
    estimators=[
        ('lr',  LogisticRegression(
            C=1.0, class_weight='balanced', max_iter=1000, random_state=42)),
        ('rf',  RandomForestClassifier(
            n_estimators=100, max_depth=15, class_weight='balanced',
            n_jobs=-1, random_state=42)),
        ('xgb', XGBClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            eval_metric='mlogloss', n_jobs=-1, random_state=42,
            use_label_encoder=False)),
    ],
    voting='soft',
    weights=[1, 2, 3]
)

pipeline = Pipeline([
    ('features',   feature_union),
    ('scaler',     MaxAbsScaler()),
    ('classifier', ensemble),
])

print('Pipeline architecture:')
print('  features   → FeatureUnion(TF-IDF + TextStats + Sentiment + LearnedVocab)')
print('  scaler     → MaxAbsScaler (preserves sparsity)')
print('  classifier → SoftVoting(LR×1, RF×2, XGB×3)')

In [ ]:
print('Training ensemble pipeline...')
t0 = time.time()
pipeline.fit(X_train, y_train_enc)
elapsed = time.time() - t0
print(f'✓ Training complete in {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)')

## 6. Model Evaluation on Test Set

In [ ]:
y_pred  = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)

accuracy  = accuracy_score(y_test_enc, y_pred)
p, r, f1, _ = precision_recall_fscore_support(
    y_test_enc, y_pred, average='weighted'
)

print('=' * 55)
print('BASE ENSEMBLE — TEST SET PERFORMANCE')
print('=' * 55)
print(f'  Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)')
print(f'  Precision : {p:.4f}')
print(f'  Recall    : {r:.4f}')
print(f'  F1 Score  : {f1:.4f}')
print()
print('Per-class report:')
print(classification_report(
    y_test_enc, y_pred,
    target_names=label_encoder.classes_,
    digits=4
))

## 7. Confusion Matrix Analysis

In [ ]:
cm = confusion_matrix(y_test_enc, y_pred)
cm_norm = confusion_matrix(y_test_enc, y_pred, normalize='true')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            ax=axes[0], linewidths=0.5)
axes[0].set_title(f'Confusion Matrix — Raw Counts\n(Accuracy: {accuracy:.2%})',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted', fontsize=11)
axes[0].set_ylabel('Actual', fontsize=11)
axes[0].tick_params(axis='x', rotation=20)

# Normalised
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            ax=axes[1], linewidths=0.5,
            vmin=0, vmax=1)
axes[1].set_title('Confusion Matrix — Normalised (Recall per class)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted', fontsize=11)
axes[1].set_ylabel('Actual', fontsize=11)
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('../outputs/figures/confusion_matrix_base.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: outputs/figures/confusion_matrix_base.png')

### Confusion Matrix — Findings

The normalised matrix (right) shows recall per class — the proportion of each actual condition that was correctly identified:

- **Depression** is classified with the highest recall — it has the most training examples and its vocabulary (sadness, hopeless, anxiety, medication names like Sertraline) is highly distinctive
- **High Blood Pressure** and **Diabetes, Type 2** show slightly lower recall — both conditions share some medical vocabulary (blood, medication, doctor) which creates occasional confusion between them
- Most misclassifications occur **between the two minority classes**, not between Depression and the others — this is the expected behaviour for an imbalanced dataset

## 8. Cross-Validation — Generalisation Assessment

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    pipeline, X_train, y_train_enc,
    cv=cv5, scoring='accuracy', n_jobs=-1
)

print('5-Fold Cross-Validation Results')
print('-' * 40)
for i, score in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'  Mean  : {cv_scores.mean():.4f}')
print(f'  Std   : {cv_scores.std():.4f}')
print(f'\n  Test accuracy  : {accuracy:.4f}')
print(f'  CV mean        : {cv_scores.mean():.4f}')
gap = accuracy - cv_scores.mean()
print(f'  Gap (test-CV)  : {gap:+.4f}')
if abs(gap) < 0.01:
    print('  → Very small gap — model generalises well, no significant overfitting')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

folds = [f'Fold {i}' for i in range(1, 6)]
bars = ax.bar(folds, cv_scores, color=PALETTE[0],
              edgecolor='white', width=0.5)
ax.axhline(cv_scores.mean(), color='#ef4444', linestyle='--', linewidth=2,
           label=f'Mean: {cv_scores.mean():.4f}')
ax.axhline(accuracy, color='#3b82f6', linestyle='--', linewidth=2,
           label=f'Test accuracy: {accuracy:.4f}')
ax.set_ylim(0.90, 1.0)
ax.set_title('5-Fold Cross-Validation Accuracy', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=11)
ax.legend(fontsize=11)
for bar, val in zip(bars, cv_scores):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('../outputs/figures/cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Ensemble vs Individual Models — Final Comparison

In [ ]:
# Add ensemble to the comparison table
ensemble_row = pd.DataFrame([{
    'Model': 'Ensemble (LR+RF+XGB) ★',
    'CV Mean': round(cv_scores.mean(), 4),
    'CV Std': round(cv_scores.std(), 4),
    'Test Accuracy': round(accuracy, 4),
    'Precision': round(p, 4),
    'Recall': round(r, 4),
    'F1 Score': round(f1, 4),
    'Time (s)': '-'
}])

full_comparison = pd.concat([results_df, ensemble_row], ignore_index=True)
full_comparison = full_comparison.sort_values(
    'Test Accuracy', ascending=False
).reset_index(drop=True)
full_comparison.index += 1

print('=== FULL MODEL COMPARISON (including ensemble) ===')
full_comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

model_names_full = full_comparison['Model'].tolist()
colors_full = ['#14b8a6' if '★' in n else '#cbd5e1' for n in model_names_full]

bars = ax.barh(model_names_full, full_comparison['Test Accuracy'],
               color=colors_full, edgecolor='white', height=0.55)
ax.set_xlim(0.85, 1.01)
ax.set_title('Final Model Comparison — Test Accuracy', fontsize=14, fontweight='bold')
ax.set_xlabel('Test Accuracy', fontsize=12)
for bar, val in zip(bars, full_comparison['Test Accuracy']):
    ax.text(float(val) + 0.001 if isinstance(val, float) else 0.951,
            bar.get_y() + bar.get_height()/2,
            f'{val}', va='center', fontsize=10, fontweight='bold')
ax.invert_yaxis()

legend_patches = [
    mpatches.Patch(color='#14b8a6', label='Selected model'),
    mpatches.Patch(color='#cbd5e1', label='Individual models'),
]
ax.legend(handles=legend_patches, fontsize=10)

plt.tight_layout()
plt.savefig('../outputs/figures/final_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Prediction Confidence Distribution

In [ ]:
max_proba = y_proba.max(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall confidence histogram
axes[0].hist(max_proba, bins=40, color='#14b8a6', edgecolor='white', alpha=0.85)
axes[0].axvline(max_proba.mean(), color='#ef4444', linestyle='--', linewidth=2,
                label=f'Mean: {max_proba.mean():.3f}')
axes[0].axvline(0.7, color='#3b82f6', linestyle=':', linewidth=2,
                label='High confidence threshold (0.7)')
axes[0].set_title('Prediction Confidence Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Max Class Probability', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].legend(fontsize=10)

# Confidence by condition
for i, (cond, color) in enumerate(
        zip(label_encoder.classes_, PALETTE)):
    mask = y_test_enc == i
    axes[1].hist(max_proba[mask], bins=30, alpha=0.65,
                 color=color, label=cond, edgecolor='white')
axes[1].set_title('Confidence by Condition', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Max Class Probability', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/figures/confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

high_conf = (max_proba >= 0.7).mean() * 100
print(f'High confidence predictions (≥0.70): {high_conf:.1f}%')
print(f'Mean confidence: {max_proba.mean():.3f}')
print(f'Min confidence : {max_proba.min():.3f}')

## 11. Training Results Summary

### Task 3 — Model Determination: Decision

After comparing five individual algorithms, the **soft-voting ensemble of Logistic Regression, Random Forest, and XGBoost** was selected as the final model. The key reasons are:

1. **Higher accuracy than any individual model** — the ensemble outperforms every individual classifier by combining their complementary strengths
2. **Error diversity** — LR makes errors on different samples than XGB; the ensemble vote resolves these disagreements
3. **Probability calibration** — soft voting preserves confidence scores used by the OOD detection system
4. **Robustness** — the low CV standard deviation (< 0.005) confirms the model generalises consistently

### Task 4 — Training Results

| Metric | Base Ensemble | After Tuning (Task 5) |
|--------|--------------|----------------------|
| Accuracy | 94.63% | **97.39%** |
| Precision | 94.76% | **97.38%** |
| Recall | 94.63% | **97.39%** |
| F1 Score | 94.54% | **97.38%** |
| CV Mean | ~94.5% | **96.61%** |

The base ensemble already exceeds 94% accuracy before any tuning, confirming the architecture choice is sound. Task 5 (hyperparameter tuning) further improved this to **97.39%**.